# Linear Algebra — AI Engineering Interview Prep

Covers: vectors, matrices, eigenvalues, SVD, PCA from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("NumPy version:", np.__version__)

## 1. Vectors

In [ ]:
a = np.array([1., 2., 3.])
b = np.array([4., 5., 6.])

print("Vector ops:")
print(f"  a + b      = {a + b}")
print(f"  3 * a      = {3 * a}")
print(f"  dot(a,b)   = {np.dot(a, b):.1f}  (= a·b = Σaᵢbᵢ)")
print(f"  cross(a,b) = {np.cross(a, b)}")
print(f"  L1 norm    = {np.linalg.norm(a, ord=1):.4f}")
print(f"  L2 norm    = {np.linalg.norm(a):.4f}")
print(f"  unit a     = {a / np.linalg.norm(a)}")

# Cosine similarity
cos_sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(f"  cos_sim    = {cos_sim:.4f}  (1=parallel, 0=orthogonal, -1=opposite)")

## 2. Matrix Operations

In [ ]:
A = np.array([[1, 2], [3, 4]], dtype=float)
B = np.array([[5, 6], [7, 8]], dtype=float)

print("Matrix A:")
print(A)
print(f"\nA @ B (matmul):\n{A @ B}")
print(f"\nA.T (transpose):\n{A.T}")
print(f"\nnp.linalg.inv(A):\n{np.linalg.inv(A)}")
print(f"\ndet(A): {np.linalg.det(A):.1f}")
print(f"\ntrace(A): {np.trace(A):.1f}  (sum of diagonal)")
print(f"\nrank(A): {np.linalg.matrix_rank(A)}")  


In [ ]:
# Solving linear system Ax = b
A = np.array([[2., 1.], [1., 3.]])
b = np.array([5., 10.])

x = np.linalg.solve(A, b)
print(f"Solution x: {x}")
print(f"Verify Ax = {A @ x}  (should equal b = {b})")

## 3. Eigenvalues & Eigenvectors

In [ ]:
# Symmetric matrix (real eigenvalues guaranteed)
A = np.array([[4., 2.], [2., 3.]])
eigenvalues, eigenvectors = np.linalg.eig(A)

print("Matrix A:")
print(A)
print(f"\nEigenvalues: {eigenvalues}")
print(f"Eigenvectors (columns):\n{eigenvectors}")

# Verify Av = λv
for i, (lam, v) in enumerate(zip(eigenvalues, eigenvectors.T)):
    Av = A @ v
    lv = lam * v
    print(f"\nv{i}: Av = {Av.round(3)}, λv = {lv.round(3)}  match: {np.allclose(Av, lv)}")

## 4. Singular Value Decomposition (SVD)

In [ ]:
# SVD: A = U @ diag(S) @ Vt
A = np.random.randn(6, 4)  # 6x4 matrix
U, S, Vt = np.linalg.svd(A, full_matrices=False)

print(f"A shape: {A.shape}")
print(f"U shape: {U.shape}  (left singular vectors)")
print(f"S shape: {S.shape}  (singular values: {S.round(3)})")
print(f"Vt shape: {Vt.shape} (right singular vectors transposed)")

# Reconstruct A
A_reconstructed = U @ np.diag(S) @ Vt
print(f"\nReconstruction error: {np.linalg.norm(A - A_reconstructed):.2e}")

# Low-rank approximation (keep top-k singular values)
k = 2
A_approx = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
variance_explained = (S[:k]**2).sum() / (S**2).sum()
print(f"Rank-{k} approximation explains {variance_explained:.1%} of variance")

## 5. PCA from Scratch

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data.astype(float)

# Step 1: Center the data
X_centered = X - X.mean(axis=0)

# Step 2: Covariance matrix
cov = np.cov(X_centered.T)  # (n_features x n_features)

# Step 3: Eigendecompose
eigenvalues, eigenvectors = np.linalg.eigh(cov)
# Sort descending
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Step 4: Project
X_pca = X_centered @ eigenvectors[:, :2]

# Explained variance
explained = eigenvalues / eigenvalues.sum()
print(f"Variance explained by each PC: {explained.round(3)}")
print(f"PC1+PC2 explain: {explained[:2].sum():.1%}")

plt.figure(figsize=(6, 4))
for label, name in enumerate(iris.target_names):
    mask = iris.target == label
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=name, alpha=0.7)
plt.xlabel(f"PC1 ({explained[0]:.1%})")
plt.ylabel(f"PC2 ({explained[1]:.1%})")
plt.title("PCA from Scratch on Iris")
plt.legend()
plt.show()

In [ ]:
# Verify: compare with sklearn PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca_sk = pca.fit_transform(X)

# May differ in sign (sign is arbitrary in PCA)
print("Explained variance ratio (sklearn):", pca.explained_variance_ratio_.round(4))
# Check correlation of projections (should be ~1 or ~-1)
for i in range(2):
    corr = np.corrcoef(X_pca[:, i], X_pca_sk[:, i])[0, 1]
    print(f"PC{i+1} correlation (scratch vs sklearn): {corr:.4f}")

## Interview Tips

- **SVD vs eigen**: SVD works on any matrix; eigen on square matrices. SVD of XᵀX/n gives PCA.
- **PCA steps**: center → covariance → eig/SVD → project. Feature scaling is crucial before PCA.
- **Singular values** vs **eigenvalues**: singular values are always non-negative; eigenvalues can be complex.
- **Rank of matrix** = number of non-zero singular values = dimension of column space.
- **Cosine similarity** in embeddings: 1=same direction, 0=orthogonal, used in semantic search.
- **Determinant = 0**: matrix is singular (no inverse), system has no unique solution.